In [ ]:
import numpy as np
import ioh


class LLM_CMAES:
    '''
    CMA-ES with Orthogonalized Mirrored Sampling, boundary reflection handling,
    and elitist mean update.
    '''
    def __init__(self,
                 pop_size: int = None,
                 num_iterations: int = 1000,
                 sigma: float = 0.5):

        self.pop_size = pop_size
        self.num_iterations = num_iterations
        self.sigma = sigma

        self.variant_description = {
            "sampling": "Orthogonalized Mirrored Sampling: first min(lambda/2 + 1, dim) directions are orthogonalized via QR decomposition to reduce redundancy in search directions, then extended randomly if necessary. Mirrored pairs (z, -z) are formed and truncated to lambda size.",
            "mean_update": "Elitist selection: weighted recombination of selected solutions is combined with the best solution from the previous generation with a small fixed weight (0.1 * weights[0]) to preserve elite solutions without fully overriding the distribution.",
            "adaptation": "Remains unchanged from the parent algorithm: standard CMA-ES sigma and covariance updates."
        }

    def __call__(self, problem: ioh.problem.RealSingleObjective):

        dim = problem.meta_data.n_variables
        low_b, high_b = problem.bounds.lb[0], problem.bounds.ub[0]

        if self.pop_size is None:
            lambda_ = 4 + int(3 * np.log(dim))
        else:
            lambda_ = self.pop_size

        mu = lambda_ // 2

        weights = np.log(mu + 0.5) - np.log(np.arange(1, mu + 1))
        weights = weights / np.sum(weights)

        mu_eff = 1.0 / np.sum(weights ** 2)

        c_sigma = (mu_eff + 2) / (dim + mu_eff + 5)

        d_sigma = (
            1
            + 2 * max(0, np.sqrt((mu_eff - 1) / (dim + 1)) - 1)
            + c_sigma
        )

        c_c = (4 + mu_eff / dim) / (dim + 4 + 2 * mu_eff / dim)

        c1 = 2 / ((dim + 1.3) ** 2 + mu_eff)

        c_mu = min(
            1 - c1,
            2 * (mu_eff - 2 + 1 / mu_eff) / ((dim + 2) ** 2 + mu_eff)
        )

        expected_norm = np.sqrt(dim) * (
            1 - 1 / (4 * dim) + 1 / (21 * dim ** 2)
        )

        mean = np.random.uniform(low_b, high_b, dim)
        sigma = self.sigma * (high_b - low_b)

        C = np.eye(dim)

        p_sigma = np.zeros(dim)
        p_c = np.zeros(dim)

        best_solution = None
        best_fitness = np.inf

        history = {
            "best_fitness": [],
            "sigma": [],
            "mean": [],
            "stagnation_counter": 0,
            "restart_count": 0
        }

        for iteration in range(self.num_iterations):

            # Numerical safeguard for covariance matrix
            C = (C + C.T) / 2

            if not np.all(np.isfinite(C)):
                C = np.eye(dim)

            C = C + 1e-12 * np.eye(dim)

            eigenvalues, B = np.linalg.eigh(C)
            eigenvalues = np.maximum(eigenvalues, 1e-20)

            D = np.sqrt(eigenvalues)
            BD = B @ np.diag(D)

            # ========================================================
            # LLM SECTION 1: Sampling
            # ========================================================

            # Orthogonalized Mirrored Sampling
            half_lambda = lambda_ // 2
            n_samples = half_lambda + 1
            k = min(n_samples, dim)

            # Generate orthogonal basis for first k directions
            A = np.random.randn(dim, k)
            Q, _ = np.linalg.qr(A)
            # Q is (dim, k), Q.T is (k, dim). Rows are orthonormal.
            # Scale to match expected norm of standard normal vectors
            z_half_ortho = Q.T * np.sqrt(dim)

            if n_samples > k:
                # Fill remaining directions with random samples
                z_half_rest = np.random.randn(n_samples - k, dim)
                z_half = np.vstack([z_half_ortho, z_half_rest])
            else:
                z_half = z_half_ortho

            # Mirrored sampling: sample lambda/2 directions and their negations
            z_full = np.vstack([z_half, -z_half])[:lambda_]

            population = mean + sigma * (z_full @ BD.T)

            # Numerical safeguard for candidate solutions
            if not np.all(np.isfinite(population)):
                population = np.nan_to_num(
                    population,
                    nan=(low_b + high_b) / 2,
                    posinf=high_b,
                    neginf=low_b
                )

            # Boundary reflection instead of clipping
            # Reflect across upper bound: if x > ub, x = ub - (x - ub)
            mask_upper = population > high_b
            excess_upper = population[mask_upper] - high_b
            population[mask_upper] = high_b - excess_upper

            # Reflect across lower bound: if x < lb, x = lb + (lb - x)
            mask_lower = population < low_b
            excess_lower = low_b - population[mask_lower]
            population[mask_lower] = low_b + excess_lower

            # ========================================================
            # END LLM SECTION 1
            # ========================================================

            fitness = np.array([problem(ind) for ind in population])

            sorted_indices = np.argsort(fitness)

            population = population[sorted_indices]
            z_full = z_full[sorted_indices]
            fitness = fitness[sorted_indices]

            previous_best_fitness = best_fitness
            previous_best_solution = best_solution

            if fitness[0] < best_fitness:
                best_fitness = fitness[0]
                best_solution = population[0].copy()
                history["stagnation_counter"] *= 0
            else:
                history["stagnation_counter"] += 1

            selected_population = population[:mu]
            selected_z = z_full[:mu]

            old_mean = mean.copy()

            # ========================================================
            # LLM SECTION 2: Mean update
            # ========================================================

            mean_offspring = np.sum(
                weights[:, np.newaxis] * selected_population,
                axis=0
            )

            # Elitist selection mechanism: combine mean offspring with previous best
            if previous_best_solution is not None:
                w_elite = 0.1 * weights[0]
                mean = (1 - w_elite) * mean_offspring + w_elite * previous_best_solution
            else:
                mean = mean_offspring

            # ========================================================
            # END LLM SECTION 2
            # ========================================================

            # ========================================================
            # LLM SECTION 3: Adaptation
            # ========================================================

            C_inv_sqrt = B @ np.diag(1 / D) @ B.T

            p_sigma = (
                (1 - c_sigma) * p_sigma
                + np.sqrt(c_sigma * (2 - c_sigma) * mu_eff)
                * (C_inv_sqrt @ ((mean - old_mean) / sigma))
            )

            sigma_exponent = (
                (c_sigma / d_sigma)
                * (np.linalg.norm(p_sigma) / expected_norm - 1)
            )

            # Prevent overflow in the exponential step-size update
            sigma_exponent = np.clip(
                sigma_exponent,
                -50.0,
                50.0
            )

            sigma = sigma * np.exp(sigma_exponent)

            # Recover from numerical instability if sigma becomes NaN or Inf
            if not np.isfinite(sigma):
                sigma = self.sigma * (high_b - low_b)

            sigma = np.clip(
                sigma,
                1e-12,
                1e3 * (high_b - low_b)
            )

            norm_p_sigma = np.linalg.norm(p_sigma)

            h_sigma_condition = norm_p_sigma / np.sqrt(
                1 - (1 - c_sigma) ** (2 * (iteration + 1))
            ) < (1.4 + 2 / (dim + 1)) * expected_norm

            h_sigma = 1.0 if h_sigma_condition else 0.0

            p_c = (
                (1 - c_c) * p_c
                + h_sigma
                * np.sqrt(c_c * (2 - c_c) * mu_eff)
                * ((mean - old_mean) / sigma)
            )

            y = (selected_population - old_mean) / sigma

            rank_mu_update = np.zeros((dim, dim))

            for i in range(mu):
                rank_mu_update += weights[i] * np.outer(y[i], y[i])

            C = (
                (1 - c1 - c_mu) * C
                + c1 * np.outer(p_c, p_c)
                + c_mu * rank_mu_update
            )

            C = (C + C.T) / 2

            # Final covariance safeguard
            if not np.all(np.isfinite(C)):
                C = np.eye(dim)

            C = C + 1e-12 * np.eye(dim)

            # ========================================================
            # END LLM SECTION 3
            # ========================================================

            history["best_fitness"].append(best_fitness)
            history["sigma"].append(sigma)
            history["mean"].append(mean.copy())

        return best_solution, best_fitness